# Mathematics and Statistics for Data Analysis – Homework 4

**Student:** Yusra Qayyum  
**Course:** Mathematics and Statistics for Data Analysis  
**Assignment:** Homework 4 — Theory + Computational Applications

This notebook contains:

- **Part 1 – Geometric Foundations and Best Approximation**  
  Conceptual questions on vector spaces, norms, inner products, Hilbert spaces,
  best approximation, Gram matrices, dual bases, and stability.

- **Part 2 – EVD, SVD, and Applications**  
  Theory questions on eigenvalue and singular value decompositions, pseudoinverse,
  instability, regularization, TLS, PCA, and the Gram operator / Riesz bases.

- **Part 3 – Computational Applications (Coding)**  
  Code demonstrations for:
  - Q13: Gram Matrix, Least Squares, and Truncated SVD
  - Q14: When Does Total Least Squares Help?
  - Q15: PCA and Low-Rank Approximation (digit image)
  - Q18: Regression with Explicit and Kernel Features

The code in this notebook **reuses the modular `.py` files in `src/`**, and
visualizations/logs are saved under `outputs/figures/` and `outputs/logs/`.


In [8]:
%cd ..

/home/saad-alam/Documents/assignments/Yusra's_stats/homework4


/home/saad-alam/Documents/assignments/Yusra's_stats/homework4/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [9]:
import numpy as np
import matplotlib.pyplot as plt
# Import the computational modules implemented in src/
from src.monomial_ls_tsvd import run_monomial_experiment
from src.tls_experiments import run_tls_experiments
from src.pca_analysis import run_pca_steps
from src.regression_kernels import run_regression_comparison

In [14]:
# Q13 – Gram Matrix, LS, and Stability via Truncated SVD
results_q13 = run_monomial_experiment()
results_q13


[23:34:54] [INFO] --- (a) LS Solution ---
[23:34:54] [INFO] a_LS = [1.01226382 0.85222614 0.83977259]
[23:34:54] [INFO] Condition number κ(G) = 4.86e+02
[23:34:54] [INFO] --- (b) TSVD Solution (R'=2) ---
[23:34:54] [INFO] Singular values of A: [8.40549389 2.51909728 0.38125914]
[23:34:54] [INFO] a_TSVD = [1.00012127 0.92119543 0.77331851]
[23:34:54] [INFO] --- (c) Coefficient Norms ---
[23:34:54] [INFO] ||a_LS||_2   = 1.5672
[23:34:54] [INFO] ||a_TSVD||_2 = 1.5642
[23:34:54] [INFO] Saved figure to outputs/figures/monomial_ls_tsvd.png
[23:34:54] [INFO] --- (d) Residual Errors ---
[23:34:54] [INFO] LS residual      = 0.039550
[23:34:54] [INFO] TSVD residual    = 0.054028


{'a_LS': array([1.01226382, 0.85222614, 0.83977259]),
 'a_TSVD': array([1.00012127, 0.92119543, 0.77331851]),
 'res_LS': np.float64(0.03955014701713053),
 'res_TSVD': np.float64(0.054027830403766464),
 'kappa_G': np.float64(486.0555972720651),
 'R_vals': [1, 2, 3],
 'residuals': [np.float64(1.0954779693372632),
  np.float64(0.054027830403766464),
  np.float64(0.03955014701713047)],
 'coeff_norms': [np.float64(1.5027354409172708),
  np.float64(1.5642458541997155),
  np.float64(1.5672222056914193)]}

### Q13 – Discussion

The Gram matrix \(G = A^\top A\) for the monomial basis \(\{1, t, t^2\}\) on
\([0,1]\) has a **large condition number** \(\kappa(G)\), reflecting the fact
that monomials are poorly conditioned as a basis (highly correlated columns).

- The ordinary LS solution \(a_{\text{LS}}\) exactly inverts the small
  singular values, so the coefficients can become large and sensitive to noise.
- The truncated SVD solution \(a_{\text{TSVD}}\) with \(R'=2\) “drops” the
  smallest singular direction, yielding a smaller coefficient norm and improved
  numerical stability, at the cost of a slightly larger residual.

The residual vs. truncation level \(R'\) reveals the **bias–variance trade-off**:
- Small \(R'\): higher bias (poorer approximation), but more stable.
- Larger \(R'\): lower bias but increased sensitivity to ill-conditioning.

In practice, one chooses \(R'\) so that most of the energy of the singular
values is retained, while discarding the noise-dominated directions.


In [11]:
# Q14 – When does TLS help?
results_q14 = run_tls_experiments()
results_q14

[23:11:03] [INFO] ==================================================
[23:11:03] [INFO] EXPERIMENT 1: Noise only in y (x is clean)
[23:11:04] [INFO] ==================================================
[23:11:04] [INFO] True theta:   2.5000
[23:11:04] [INFO] OLS estimate: 2.4455 (error = 0.0545)
[23:11:04] [INFO] TLS estimate: 2.4567 (error = 0.0433)
[23:11:04] [INFO] ==================================================
[23:11:04] [INFO] EXPERIMENT 2: Noise in both x and y
[23:11:04] [INFO] ==================================================
[23:11:04] [INFO] True theta:   2.5000
[23:11:04] [INFO] OLS estimate: 2.4226 (error = 0.0774)
[23:11:04] [INFO] TLS estimate: 2.5105 (error = 0.0105)
[23:11:04] [INFO] Saved figure to outputs/figures/tls_experiments.png


{'theta_true': 2.5,
 'exp1': (2.4455063556930843,
  2.4567233922945806,
  0.05449364430691572,
  0.04327660770541941),
 'exp2': (2.4226328044936127,
  2.510535293503578,
  0.07736719550638727,
  0.010535293503577847)}

### Q14 – Discussion: When Does TLS Help?

In **Experiment 1**, only the output \(y\) is corrupted by noise, while the
input \(x\) is exact. LS is designed exactly for this model and tends to
perform better than TLS. TLS may over-correct by trying to adjust both
\(x\) and \(y\), even though the true \(x\) is noise-free, leading to a larger
parameter error \(|\hat\theta_{\text{TLS}} - \theta_{\text{true}}|\).

In **Experiment 2**, both \(x\) and \(y\) are corrupted with noise of similar
magnitude. The LS model is now mismatched (it still assumes exact \(x\)), so
it typically underestimates the uncertainty and can be biased. TLS, which
explicitly allows perturbations in both \(A\) (inputs) and \(y\), is more
appropriate and often yields a smaller parameter error.

In summary:
- LS is optimal (in a least-squares sense) when the design matrix is exact and
  noise is only in \(y\).
- TLS becomes advantageous when measurement errors affect **both** inputs and
  outputs, matching the TLS error model.

In [12]:
# Q15 – PCA and Low-Rank Approximation of a Digit Image
results_q15 = run_pca_steps()
results_q15


[23:11:29] [INFO] ============================================================
[23:11:29] [INFO] PCA Step-by-Step: Geometry, Decorrelation, Compression
[23:11:29] [INFO] ============================================================
[23:11:29] [INFO] Image shape: (16, 16), Total values: 256
[23:11:29] [INFO] ---- (a) Geometric Transformation ----
[23:11:29] [INFO] First principal component v1: [ 0.00000000e+00 -2.22044605e-16  0.00000000e+00  6.93889390e-18
  3.84609246e-01  3.84609246e-01  4.25722734e-01  4.25722734e-01
  4.63018339e-01  3.56771329e-01  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
[23:11:29] [INFO] Saved PCA step figure to outputs/figures/pca_steps.png
[23:11:29] [INFO] ---- (b) Statistical Decorrelation ----
[23:11:29] [INFO] Off-diagonal energy: Original = 30695.07, PCA = 0.00e+00
[23:11:30] [INFO] Saved covariance heatmaps to outputs/figures/pca_covariance_heatmaps.png
[23:11:30] [INFO] ---- (c) Dimensionality Reduc

{'errors_svd': {1: np.float64(603.0686128922605),
  3: np.float64(154.6542262870419),
  5: np.float64(6.720030319178291e-13),
  10: np.float64(6.73027117740753e-13),
  16: np.float64(6.73027117740753e-13)},
 'error_cov': np.float64(154.65422628704192),
 'diff': np.float64(9.86043970341699e-13),
 'off_orig': np.float64(30695.072894521687),
 'off_pca': np.float64(0.0)}

### Q15 – Discussion: PCA, Decorrelation, and Compression

The SVD-based PCA of the \(16\times16\) digit image shows:

- **Geometric transformation:**  
  Rotating the data into the principal component basis (matrix \(Z\)) aligns the
  coordinate axes with directions of maximum variance. The first principal
  component vector \(v_1\) often corresponds to the dominant stroke pattern of
  the digit.

- **Statistical decorrelation:**  
  The covariance of the original centered data \(C_{\text{orig}}\) has
  significant off-diagonal energy, indicating correlated columns (pixels).
  After rotation, the covariance \(C_{\text{pca}}\) becomes nearly diagonal:
  off-diagonal energy is greatly reduced. PCA thus decorrelates the data.

- **Compression:**  
  Using only the top \(k=3\) principal components gives a reconstruction with
  relatively small Frobenius error \(\|X - \hat X\|_F\) compared to the original
  energy. The storage cost drops from 256 numbers to \(17k+16=67\), giving a
  substantial compression ratio while preserving the main structure of the digit.

- **Covariance vs. SVD method:**  
  Implementing PCA via eigendecomposition of the covariance matrix
  \(C = \frac{1}{n}\tilde{X}^\top\tilde{X}\) yields essentially the same
  reconstructed image as the direct SVD method (for \(k=3\)), and the numerical
  difference is on the order of machine precision. This confirms that the two
  formulations are theoretically equivalent.


In [13]:
# Q18 – Regression with Explicit and Kernel Methods
results_q18 = run_regression_comparison()
results_q18

[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] REGRESSION COMPARISON: Explicit vs. Kernel Methods
[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] True function: y = 2*sin(2x) + 0.5*x^3
[23:11:55] [INFO] Samples: 50, Noise std: 0.5
[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] MEAN SQUARED ERROR (MSE) vs. TRUE FUNCTION
[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] Method                    MSE         
[23:11:55] [INFO] ------------------------------------------------------------
[23:11:55] [INFO] Linear OLS                3.6367e+00  
[23:11:55] [INFO] Gramian (Linear)          3.6372e+00  
[23:11:55] [INFO] Polynomial (deg 5)        1.8155e-01  
[23:11:55] [INFO] RBF Kernel                2.8419e-01  
[23:11:55] [INFO] Best method: Polynomial (deg 5) (MSE = 1.8155e-01)
[23:11:55] [INFO] 

{'mse_ols': 3.6367461199253,
 'mse_poly': 0.18155439686612188,
 'mse_gram_lin': 3.6371733720051616,
 'mse_rbf': 0.2841922590985657,
 'best_method': 'Polynomial (deg 5)'}